In [1]:
import time
import numpy as np
import pandas as pd
import gurobipy as gp

from gurobipy import GRB

In [2]:
def solve_or_milp_indicator(
    df_day,
    lam,
    n_rooms=3,
    room_capacity=480,
    turnover=20,
    balance_weight=0.10,
    duration_cap=360.0,
    time_limit=300,
):

    # ============================================================
    # Basic data
    # ============================================================

    N_CASES = len(df_day)

    if N_CASES == 0:
        raise ValueError("df_day must contain at least one surgery.")

    I = range(N_CASES)
    R = range(n_rooms)

    P50 = df_day["DURATION_P50_MINS"].values
    P90 = df_day["DURATION_P90_MINS"].values


    # ============================================================
    # Risk-adjusted duration
    # ============================================================

    duration_uncapped = (
        P50
        + lam * (P90 - P50)
    )

    # Apply the prespecified duration cap for the main experiment.
    # Set duration_cap=None for the uncapped sensitivity analysis.

    if duration_cap is None:

        duration = duration_uncapped.copy()

        n_capped = 0
        pct_capped = 0.0

    else:

        if duration_cap <= 0:
            raise ValueError(
                "duration_cap must be positive or None."
            )

        cap_mask = duration_uncapped > duration_cap

        n_capped = int(
            np.sum(cap_mask)
        )

        pct_capped = float(
            100.0 * n_capped / N_CASES
        )

        duration = np.minimum(
            duration_uncapped,
            duration_cap
        )


    # ============================================================
    # Safe instance-specific horizon
    # ============================================================

    SERIAL_HORIZON = float(
        np.sum(duration)
        + turnover * max(N_CASES - 1, 0)
    )

    # Retain the original start-time horizon for a fair A/B test.
    DAY_END = 1440


    # ============================================================
    # Big-M
    #
    # Sequencing itself does NOT use Big-M in this test.
    # M is retained only for the room-finishing constraints so that
    # the rest of the formulation remains identical to the baseline.
    # ============================================================

    max_duration = float(
        np.max(duration)
    )

    M = float(
        DAY_END
        + max_duration
        + turnover
    )


    # ============================================================
    # Build Gurobi model
    # ============================================================

    m = gp.Model(
        f"OR_indicator_lambda_{lam:.2f}"
    )

    m.setParam(
        "OutputFlag",
        0
    )

    m.setParam(
        "MIPGap",
        0.01
    )

    m.setParam(
        "TimeLimit",
        time_limit
    )

    m.setParam(
        "Heuristics",
        0.4
    )


    # ============================================================
    # Decision variables
    # ============================================================

    # x[i,r] = 1 if surgery i is assigned to room r

    x = m.addVars(
        I,
        R,
        vtype=GRB.BINARY,
        name="x"
    )


    # s[i] = start time of surgery i

    s = m.addVars(
        I,
        lb=0,
        ub=DAY_END,
        vtype=GRB.CONTINUOUS,
        name="start"
    )


    # c[i] = completion time of surgery i

    c = m.addVars(
        I,
        lb=0,
        vtype=GRB.CONTINUOUS,
        name="completion"
    )


    # C[r] = finishing time of the last surgery in room r

    C = m.addVars(
        R,
        lb=0,
        ub=SERIAL_HORIZON,
        vtype=GRB.CONTINUOUS,
        name="room_finish"
    )


    # Overtime for room r

    OT = m.addVars(
        R,
        lb=0,
        vtype=GRB.CONTINUOUS,
        name="overtime"
    )


    # Idle time for room r

    IDLE = m.addVars(
        R,
        lb=0,
        vtype=GRB.CONTINUOUS,
        name="idle"
    )


    # Overall system makespan

    system_makespan = m.addVar(
        lb=0,
        ub=SERIAL_HORIZON,
        vtype=GRB.CONTINUOUS,
        name="makespan"
    )


    # Minimum room completion time

    min_room_load = m.addVar(
        lb=0,
        ub=SERIAL_HORIZON,
        vtype=GRB.CONTINUOUS,
        name="min_load"
    )


    # ============================================================
    # Sequencing variables — indicator formulation
    # ============================================================
    #
    # same_room[i,j,r] = 1 iff both i and j are assigned to room r
    #
    # before_ij[i,j,r] = 1 iff:
    #     i and j are both in room r AND i is before j
    #
    # before_ji[i,j,r] = 1 iff:
    #     i and j are both in room r AND j is before i
    #
    # For a same-room pair:
    #     before_ij + before_ji = 1
    #
    # For a different-room pair:
    #     before_ij = before_ji = 0
    #
    # ============================================================

    same_room = {}
    before_ij = {}
    before_ji = {}

    for i in I:

        for j in I:

            if i >= j:
                continue

            for r in R:

                same_room[i, j, r] = m.addVar(
                    vtype=GRB.BINARY,
                    name=f"same_room_{i}_{j}_{r}"
                )

                before_ij[i, j, r] = m.addVar(
                    vtype=GRB.BINARY,
                    name=f"before_ij_{i}_{j}_{r}"
                )

                before_ji[i, j, r] = m.addVar(
                    vtype=GRB.BINARY,
                    name=f"before_ji_{i}_{j}_{r}"
                )


    # ============================================================
    # Assignment constraints
    # ============================================================

    # Every surgery must be assigned to exactly one OR.

    for i in I:

        m.addConstr(
            gp.quicksum(
                x[i, r]
                for r in R
            ) == 1,
            name=f"assignment_{i}"
        )


    # ============================================================
    # Sequencing constraints — indicator formulation
    # ============================================================

    for i in I:

        for j in I:

            if i >= j:
                continue

            for r in R:

                a = same_room[i, j, r]
                bij = before_ij[i, j, r]
                bji = before_ji[i, j, r]


                # ------------------------------------------------
                # same_room = x[i,r] AND x[j,r]
                # ------------------------------------------------

                m.addConstr(
                    a <= x[i, r],
                    name=f"same_room_i_{i}_{j}_{r}"
                )

                m.addConstr(
                    a <= x[j, r],
                    name=f"same_room_j_{i}_{j}_{r}"
                )

                m.addConstr(
                    a >= x[i, r] + x[j, r] - 1,
                    name=f"same_room_and_{i}_{j}_{r}"
                )


                # ------------------------------------------------
                # Exactly one ordering direction when same-room.
                #
                # If a = 1:
                #     bij + bji = 1
                #
                # If a = 0:
                #     bij = bji = 0
                # ------------------------------------------------

                m.addConstr(
                    bij + bji == a,
                    name=f"order_select_{i}_{j}_{r}"
                )


                # ------------------------------------------------
                # Indicator: i before j
                # ------------------------------------------------

                m.addGenConstrIndicator(
                    bij,
                    True,
                    s[i] + duration[i] + turnover <= s[j],
                    name=f"indicator_ij_{i}_{j}_{r}"
                )


                # ------------------------------------------------
                # Indicator: j before i
                # ------------------------------------------------

                m.addGenConstrIndicator(
                    bji,
                    True,
                    s[j] + duration[j] + turnover <= s[i],
                    name=f"indicator_ji_{i}_{j}_{r}"
                )


    # ============================================================
    # Completion identity
    # ============================================================

    for i in I:

        m.addConstr(
            c[i]
            ==
            s[i] + duration[i],
            name=f"completion_{i}"
        )


    # ============================================================
    # Room finishing time
    # ============================================================
    #
    # C[r] must be at least the completion time of every surgery
    # assigned to room r.
    #
    # This part intentionally retains the baseline Big-M structure
    # so that the A/B test changes only the sequencing formulation.
    #
    # No turnover is added after the final surgery.
    # ============================================================

    for r in R:

        for i in I:

            m.addConstr(
                C[r]
                >=
                c[i]
                -
                M * (1 - x[i, r]),
                name=f"room_finish_{i}_{r}"
            )


    # ============================================================
    # Room symmetry breaking
    # ============================================================

    for r in range(n_rooms - 1):

        m.addConstr(
            C[r] >= C[r + 1],
            name=f"room_symmetry_{r}"
        )


    # ============================================================
    # System makespan and load balancing
    # ============================================================

    for r in R:

        # Overall makespan >= each room's finishing time

        m.addConstr(
            system_makespan
            >=
            C[r],
            name=f"makespan_room_{r}"
        )

        # Minimum room load <= each room's finishing time

        m.addConstr(
            min_room_load
            <=
            C[r],
            name=f"min_load_room_{r}"
        )


    # ============================================================
    # Valid workload lower bound
    # ============================================================

    total_duration = float(
        np.sum(duration)
    )

    m.addConstr(
        system_makespan
        >=
        total_duration / n_rooms,
        name="makespan_workload_lb"
    )


    # ============================================================
    # Overtime / idle time relationship
    # ============================================================
    #
    # OT[r] - IDLE[r] = C[r] - room_capacity
    #
    # At an optimal incumbent, positive objective coefficients make:
    #
    # C[r] > capacity:
    #     OT[r] = C[r] - capacity
    #
    # C[r] < capacity:
    #     IDLE[r] = capacity - C[r]
    #
    # ============================================================

    for r in R:

        m.addConstr(
            OT[r]
            -
            IDLE[r]
            ==
            C[r]
            -
            room_capacity,
            name=f"shift_balance_{r}"
        )


    # ============================================================
    # Objective function
    # ============================================================

    objective = (

        # Total overtime

        gp.quicksum(
            OT[r]
            for r in R
        )

        +

        # Idle-time penalty

        0.25
        *
        gp.quicksum(
            IDLE[r]
            for r in R
        )

        +

        # Makespan penalty

        0.5
        *
        system_makespan

        +

        # Workload balancing penalty

        balance_weight
        *
        (
            system_makespan
            -
            min_room_load
        )
    )

    m.setObjective(
        objective,
        GRB.MINIMIZE
    )


    # ============================================================
    # Solve
    # ============================================================

    solve_start = time.time()

    m.optimize()

    solve_time = (
        time.time()
        -
        solve_start
    )


    # ============================================================
    # Solver diagnostics
    # ============================================================

    solver_status = int(
        m.Status
    )

    hit_time_limit = (
        solver_status == GRB.TIME_LIMIT
    )

    if m.SolCount > 0:

        final_mip_gap = float(
            m.MIPGap
        )

        reached_1pct_gap = (
            final_mip_gap <= 0.01 + 1e-9
        )

    else:

        final_mip_gap = np.nan
        reached_1pct_gap = False


    # ============================================================
    # Check solution
    # ============================================================

    if m.SolCount == 0:

        return None


    # ============================================================
    # Metrics
    # ============================================================

    overtime = sum(
        OT[r].X
        for r in R
    )

    idle = sum(
        IDLE[r].X
        for r in R
    )

    total_schedule = sum(
        C[r].X
        for r in R
    )


    # ------------------------------------------------------------
    # Schedule utilization
    #
    # Retains the original baseline definition:
    # total P50 surgical time / total room finishing time
    # ------------------------------------------------------------

    schedule_utilization = (
        P50.sum()
        /
        total_schedule
    )


    # ------------------------------------------------------------
    # Capacity utilization
    # ------------------------------------------------------------

    capacity_utilization = (
        P50.sum()
        /
        (
            room_capacity
            *
            n_rooms
        )
    )


    # ------------------------------------------------------------
    # Capacity expansion
    # ------------------------------------------------------------

    capacity_expansion = (
        total_schedule
        /
        (
            room_capacity
            *
            n_rooms
        )
    )


    # ------------------------------------------------------------
    # On-time room rate
    # ------------------------------------------------------------

    ontime_rate = (
        sum(
            1
            for r in R
            if C[r].X <= room_capacity
        )
        /
        n_rooms
    )


    # ------------------------------------------------------------
    # Load gap
    # ------------------------------------------------------------

    load_gap = (
        system_makespan.X
        -
        min_room_load.X
    )


    # ============================================================
    # Recover sequencing solution
    # ============================================================

    sequencing_solution = []

    for i in I:

        for j in I:

            if i >= j:
                continue

            for r in R:

                if before_ij[i, j, r].X > 0.5:

                    sequencing_solution.append(
                        {
                            "room": r,
                            "first": i,
                            "second": j
                        }
                    )

                elif before_ji[i, j, r].X > 0.5:

                    sequencing_solution.append(
                        {
                            "room": r,
                            "first": j,
                            "second": i
                        }
                    )


    # ============================================================
    # Return dictionary
    # ============================================================

    return {

        "Lambda":
            lam,

        "Time_Limit":
            time_limit,

        "Duration_Cap":
            duration_cap,

        "N_Capped":
            n_capped,

        "Pct_Capped":
            pct_capped,

        "Max_Uncapped_Duration":
            float(
                np.max(duration_uncapped)
            ),

        "Max_Planning_Duration":
            float(
                np.max(duration)
            ),

        "Objective":
            m.objVal,

        "Overtime":
            overtime,

        "Idle":
            idle,

        "Makespan":
            system_makespan.X,

        "Sched_Util":
            schedule_utilization,

        "Cap_Util":
            capacity_utilization,

        "Buffer_Ratio":
            duration.sum()
            /
            P50.sum(),

        "Cap_Expansion":
            capacity_expansion,

        "OnTime_Rate":
            ontime_rate,

        "Load_Gap":
            load_gap,

        "Solve_Time":
            solve_time,

        "Variables":
            m.NumVars,

        "Constraints":
            m.NumConstrs,

        "General_Constraints":
            m.NumGenConstrs,

        "Nodes":
            m.NodeCount,

        "Gap":
            final_mip_gap,

        "Status":
            solver_status,

        "Reached_1pct_Gap":
            reached_1pct_gap,

        "Hit_Time_Limit":
            hit_time_limit,

        "sequencing":
            sequencing_solution,

        "Start_Times":
            {
                i: s[i].X
                for i in I
            },

        "room_assignment":
            {
                i: next(
                    r
                    for r in R
                    if x[i, r].X > 0.5
                )
                for i in I
            },

        "start_times":
            {
                i: s[i].X
                for i in I
            }
    }

In [3]:
# ============================================================
# INDICATOR MILP A/B TEST
# LightGBM | Fixed 12-case cohort | lambda = 0.5
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Load LightGBM optimizer input
# ------------------------------------------------------------

lightgbm_df = pd.read_csv(
    "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv"
)

print(
    "LightGBM optimizer input:",
    lightgbm_df.shape
)


# ------------------------------------------------------------
# 2. Load fixed 12-case cohort
# ------------------------------------------------------------

cohort = pd.read_csv(
    "Cohort/fixed_cohort.csv"
)

print(
    "Fixed cohort:",
    cohort.shape
)


# ------------------------------------------------------------
# 3. Match fixed cohort
# ------------------------------------------------------------

cohort_ids = cohort["LOG_ID"].tolist()

lightgbm_fixed = (
    lightgbm_df
    .set_index("LOG_ID")
    .loc[cohort_ids]
    .reset_index()
)

assert len(lightgbm_fixed) == 12

print(
    "Matched LightGBM fixed cohort:",
    lightgbm_fixed.shape
)

print(
    lightgbm_fixed[
        [
            "LOG_ID",
            "DURATION_P50_MINS",
            "DURATION_P90_MINS"
        ]
    ]
)


# ------------------------------------------------------------
# 4. Run indicator formulation
# ------------------------------------------------------------

test_result = solve_or_milp_indicator(
    df_day=lightgbm_fixed,
    lam=0.5,
    n_rooms=3,
    room_capacity=480,
    turnover=20,
    balance_weight=0.10,
    duration_cap=360.0,
    time_limit=300
)

assert test_result is not None, (
    "Indicator MILP returned no feasible solution."
)


# ------------------------------------------------------------
# 5. Display scalar diagnostics
# ------------------------------------------------------------

exclude_keys = {
    "sequencing",
    "Start_Times",
    "room_assignment",
    "start_times"
}

print("\n" + "=" * 70)
print("INDICATOR MILP A/B TEST RESULT")
print("=" * 70)

for key, value in test_result.items():

    if key not in exclude_keys:

        print(
            f"{key:30s}: {value}"
        )

LightGBM optimizer input: (17083, 4)
Fixed cohort: (12, 1)
Matched LightGBM fixed cohort: (12, 4)
              LOG_ID  DURATION_P50_MINS  DURATION_P90_MINS
0   f42047cb579b3055         139.360367         336.469114
1   28a358995e41a9f5         197.526428         442.002814
2   b29ebd9769926496         176.909439         359.415310
3   de7a28cae633f05a         219.187313         372.397211
4   564d6a7a6173eed1         227.901180         322.776881
5   a8da758576964b78         177.001851         362.748122
6   5a8e1df80d6b047f         162.770886         313.035974
7   ec5cecc91ac56559         165.733397         328.976841
8   cf2dd7eb2a698665         172.957740         394.432739
9   4cc0c8e2dd8ffc9c         177.120714         359.624973
10  d6b5106aa593a6fc         165.707943         322.776881
11  fc7299c4e0060a00         169.393854         346.446941
Set parameter Username
Set parameter LicenseID to value 2835369
Academic license - for non-commercial use only - expires 2027-06-15

IN